# 0. Environment Setup and Reproducibility

This project uses a Conda environment to manage its Python dependencies.

## Installation

1. Clone the project repository and navigate to its root directory:

```bash
git clone https://github.com/Gleisongarcia/Machine-learning-project.git
cd Machine-learning-project
```

2. Create the Conda environment using the provided `environment.yml` file:

```bash
conda env create -f environment.yml
```

3. Activate the environment:

```bash
conda activate smilesproject
```

4. Launch Jupyter:

```bash
jupyter notebook
```

5. Open this notebook and select the `cyp2c19-ml` kernel if necessary.

## Execution

Run all notebook cells in order, from beginning to end. An internet connection may be required to download the dataset.

The `environment.yml` file specifies the Python version and the dependencies required to run the project.


# 1. Problem Definition

## 1.1 Biological Background: Drug Metabolism and CYP2C19

Drug metabolism is a fundamental process in pharmacology. After a drug enters the body, it can be transformed by enzymatic systems that influence its elimination, duration of action and systemic exposure. Among these systems, the cytochrome P450 (CYP) family plays a major role in the metabolism of many therapeutic compounds.

CYP2C19 is one of these enzymes. It is located primarily in the endoplasmic reticulum of liver cells and participates in the metabolism of several drugs, including omeprazole and diazepam. Its activity is therefore relevant to how the body processes certain medications.

An important pharmacological phenomenon is **enzyme inhibition**. When a molecule inhibits CYP2C19, it reduces the enzyme's ability to metabolize its substrates. If another drug depends on this enzyme for its metabolism, inhibition can alter that drug's exposure in the body. Depending on the drugs involved, this may increase the risk of adverse reactions or reduce therapeutic efficacy.

Drug–drug interactions are therefore an important consideration during drug development. Identifying molecules that may inhibit drug-metabolizing enzymes can help researchers characterize potential metabolic interactions at an early stage.

## 1.2 Research Question and Objective

The central question of this project is:

**Can we predict whether a molecule inhibits CYP2C19 from its chemical structure?**

Experimental characterization of drug–enzyme interactions requires laboratory assays. Machine learning offers a complementary approach by learning statistical relationships between molecular representations and experimentally assigned labels.

The objective of this project is to develop and compare machine learning models that predict CYP2C19 inhibition from molecular structure, represented by SMILES strings.

This is a relevant problem because molecular structure influences how a compound interacts with biological targets. However, the relationship between chemical structure and enzyme inhibition is complex and cannot always be captured by simple rules.

## 1.3 Dataset and Prediction Task

This project uses the `CYP2C19_Veith` dataset provided by Therapeutics Data Commons (TDC), based on the work of Veith et al. The dataset contains 12,665 molecules and is designed as a binary classification task.

Each molecule is represented by a SMILES string (*Simplified Molecular Input Line Entry System*), a textual notation that encodes its chemical structure. The model uses this representation to derive numerical features and predict the associated inhibition label.

The target variable is `Y`, defined as follows:

- `Y = 0`: the molecule does not inhibit CYP2C19 according to the dataset label.
- `Y = 1`: the molecule inhibits CYP2C19 according to the dataset label.

The task is therefore to predict one of two classes for each molecule.

## 1.4 Machine Learning Strategy

To investigate how molecular representation affects predictive performance, this project compares two feature engineering approaches:

- **Morgan fingerprints:** fixed-length binary vectors that encode local molecular environments.
- **SMILES character n-grams:** numerical representations based on recurring sequences of characters in SMILES strings.

Both representations will be used to train classification models, including Logistic Regression and Random Forest. Their performance will be compared using appropriate classification metrics, followed by hyperparameter optimization and a final evaluation on a held-out test set.

The aim is to assess which combination of molecular representation and model performs best on this dataset, while critically examining the limitations of the predictions.



# 2. Dataset Description and Exploratory Data Analysis

## 2.1 Dataset Overview

The dataset used in this project is `CYP2C19_Veith`, available through Therapeutics Data Commons (TDC). It contains 12,665 molecules and is designed for the prediction of CYP2C19 inhibition.

Each observation corresponds to a molecule described by three main variables:

- `Drug_ID`: identifier of the molecule.
- `Drug`: molecular structure encoded as a SMILES string.
- `Y`: binary target indicating CYP2C19 inhibition (`0`: non-inhibitor, `1`: inhibitor).

Before training machine learning models, the dataset must be explored to understand its structure, assess data quality and identify potential issues that could affect the analysis.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tdc.single_pred import ADME

# Load the complete dataset
data = ADME(name="CYP2C19_Veith")
df = data.get_data()

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five observations:")
display(df.head())

print("\nData types:")
print(df.dtypes)


ModuleNotFoundError: No module named 'tdc'


## 2.2 Data Quality and SMILES Validation

Data quality is an important first step because missing values, duplicate observations or invalid molecular representations may affect feature extraction and model training.

SMILES strings will be checked using RDKit, a cheminformatics library that can parse molecular structures from their textual representations.

In [ ]:

from rdkit import Chem

# Missing values
print("Missing values per column:")
print(df.isna().sum())

# Duplicate observations
print("\nNumber of duplicated rows:", df.duplicated().sum())

# Duplicate molecule identifiers and SMILES
print("Unique Drug_ID values:", df["Drug_ID"].nunique())
print("Unique SMILES strings:", df["Drug"].nunique())

# Validate SMILES using RDKit
smiles = df["Drug"]

valid_smiles = smiles.apply(
    lambda s: Chem.MolFromSmiles(s) is not None
    if isinstance(s, str) and s.strip()
    else False
)

print("\nSMILES validation:")
print("Valid:", valid_smiles.sum())
print("Invalid:", (~valid_smiles).sum())
print("Validity rate: {:.2f}%".format(valid_smiles.mean() * 100))



## 2.3 Target Distribution

The target distribution describes the proportion of inhibitors and non-inhibitors in the dataset. This analysis helps determine whether the classification problem is strongly imbalanced, which can influence model evaluation and the choice of performance metrics.


In [ ]:

# Count observations in each class
class_counts = df["Y"].value_counts().sort_index()
class_percentages = df["Y"].value_counts(normalize=True).sort_index() * 100

target_summary = pd.DataFrame({
    "Molecule count": class_counts,
    "Percentage (%)": class_percentages
})

display(target_summary)

# Plot class distribution
plt.figure(figsize=(6, 4))
plt.bar(
    class_counts.index.astype(str),
    class_counts.values
)

plt.xlabel("Target class (Y)")
plt.ylabel("Number of molecules")
plt.title("Distribution of CYP2C19 inhibition labels")
plt.tight_layout()
plt.show()



## 2.4 SMILES Length Distribution

SMILES strings vary in length depending on how molecular structures are encoded. Examining their length distribution provides an overview of the input data and helps identify unusually long representations.

String length should not be interpreted as a direct measure of molecular complexity, since different SMILES strings can encode the same molecular structure.


In [ ]:

# Calculate SMILES string lengths
df["smiles_length"] = df["Drug"].apply(
    lambda s: len(s) if isinstance(s, str) else np.nan
)

print("SMILES length statistics:")
display(df["smiles_length"].describe())

# Plot overall distribution
plt.figure(figsize=(8, 4))
plt.hist(
    df["smiles_length"].dropna(),
    bins=40
)

plt.xlabel("SMILES length (characters)")
plt.ylabel("Number of molecules")
plt.title("Distribution of SMILES lengths")
plt.tight_layout()
plt.show()



## 2.5 Main Findings

The exploratory analysis provides several observations about the dataset:

- The dataset contains 12,665 molecules described by their identifiers, SMILES strings and binary target labels.
- Data quality was assessed by checking missing values, duplicate observations and SMILES validity.
- The two target classes are relatively balanced, so the classification task does not exhibit a strong class imbalance.
- SMILES string lengths show substantial variability, with some representations considerably longer than the median.

These findings provide the basis for the next step: transforming molecular structures into numerical features suitable for machine learning. Two representations will be investigated: Morgan fingerprints and character n-grams.


## 3. Data Preparation and Feature Engineering

Machine learning algorithms require molecules to be represented as numerical features. However, the dataset describes each molecule using a SMILES string, which is a textual representation of its chemical structure.

In this project, two complementary molecular representations are compared:

1. **Morgan fingerprints**, which encode local molecular environments as a fixed-length binary vector.
2. **SMILES character n-grams**, which encode the presence of short character sequences in the SMILES representation.

To ensure a fair comparison, both representations must use the same training, validation and test molecules.


### 3.1. Train, Validation and Test Split

The dataset is divided into three subsets:

- **Training set (70%)**: used to fit the machine learning models.
- **Validation set (10%)**: used to compare models and select hyperparameters.
- **Test set (20%)**: reserved for the final evaluation of the selected models.

A fixed random seed (`42`) is used to make the random split reproducible. The split must be created once and reused for both molecular representations, ensuring that differences in performance are not caused by evaluating the models on different molecules.

The same target labels are retained across both representations: `0` indicates a non-inhibitor and `1` indicates a CYP2C19 inhibitor.


In [ ]:
splits = data.get_split(
    method="random",
    seed=42,
    frac=[0.7, 0.1, 0.2]
)

train_df = splits["train"].copy()
valid_df = splits["valid"].copy()
test_df = splits["test"].copy()

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

### 3.2. Molecular Representation 1: Morgan Fingerprints

Morgan fingerprints represent molecules through their local chemical environments. Starting from each atom, the algorithm identifies neighboring atoms and bonds within a specified radius and converts these environments into a fixed-length fingerprint.

In this project, Morgan fingerprints are generated using **radius 2** and **2048 bits**.

Each molecule is represented by a binary vector of length 2048:

- `1` indicates that at least one molecular environment is mapped to the corresponding bit.
- `0` indicates that no environment is mapped to that bit.

**Why this representation?**

Morgan fingerprints are widely used in cheminformatics because they provide a compact numerical description of molecular structure. They encode information about local atomic environments without requiring the model to process a textual string directly.

The fixed-length representation is suitable for classical machine learning algorithms such as Logistic Regression and Random Forest. However, because the environments are hashed into a limited number of bits, different environments can occasionally map to the same bit.


In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem

def smiles_to_morgan(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=radius,
        nBits=n_bits
    )

    return np.asarray(fp, dtype=np.uint8)

X_train_morgan = np.stack(
    train_df["Drug"].map(smiles_to_morgan).to_numpy()
)
X_valid_morgan = np.stack(
    valid_df["Drug"].map(smiles_to_morgan).to_numpy()
)
X_test_morgan = np.stack(
    test_df["Drug"].map(smiles_to_morgan).to_numpy()
)

y_train = train_df["Y"].to_numpy()
y_valid = valid_df["Y"].to_numpy()
y_test = test_df["Y"].to_numpy()

print("Morgan train:", X_train_morgan.shape)
print("Morgan validation:", X_valid_morgan.shape)
print("Morgan test:", X_test_morgan.shape)

<div style="display: flex; gap: 15px;">
  <img src="figures/figures_fingerprints/active_bits_distribution.png" width="48%">
  <img src="figures/figures_fingerprints/morgan_bit_314" width="48%">
</div>

### 3.3. Molecular Representation 2: SMILES Character N-grams

The second approach represents each molecule through character n-grams extracted from its SMILES string.

An n-gram is a consecutive sequence of characters. In this project, character sequences of lengths 2 to 4 are considered. For example, a SMILES string such as `CCO` contains the 2-grams `CC` and `CO`, as well as the 3-gram `CCO`.

A `CountVectorizer` is used with the following settings:

- `analyzer="char"`: extracts character sequences rather than words.
- `ngram_range=(2, 4)`: considers sequences of 2, 3 and 4 characters.
- `min_df=2`: excludes n-grams appearing in fewer than two documents in the training data.
- `max_features=30000`: sets an upper limit on the vocabulary size.
- `binary=True`: records whether an n-gram is present, rather than how many times it occurs.

The vocabulary is learned from the training set only. The same fitted vectorizer is then applied to the validation and test sets.

**Why this representation?**

SMILES n-grams provide a complementary approach to Morgan fingerprints. Instead of explicitly constructing local molecular environments, they learn patterns from the character sequences used to encode chemical structures.

This makes the approach straightforward to implement with standard text-processing tools. However, SMILES strings are not natural language, and character sequences do not always correspond directly to chemically meaningful substructures. Results can also depend on the chosen SMILES representation.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    analyzer="char",
    ngram_range=(2, 4),
    min_df=2,
    max_features=30000,
    binary=True,
    dtype=np.float32
)

X_train_text = vectorizer.fit_transform(train_df["Drug"])
X_valid_text = vectorizer.transform(valid_df["Drug"])
X_test_text = vectorizer.transform(test_df["Drug"])

print("SMILES n-grams")
print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Train:", X_train_text.shape)
print("Validation:", X_valid_text.shape)
print("Test:", X_test_text.shape)

<div style="display: flex; gap: 15px;">
  <img src="figures/figures_text/text_active_ngrams_distribution" width="48%">
  <img src="figures/figures_text/text_frequent_ngrams" width="48%">
</div>

### 3.4. Comparison of Molecular Representations

Both approaches encode the same molecules but describe their structures differently.

| Property | Morgan fingerprints | SMILES character n-grams |
|---|---|---|
| Input | SMILES converted into molecular graphs | SMILES strings |
| Feature extraction | Local atomic environments | Consecutive character sequences |
| Feature representation | Binary vector | Binary n-gram presence vector |
| Feature dimension | 2048 | Learned vocabulary size |
| Main advantage | Compact representation of local chemical environments | Simple, data-driven extraction of textual patterns |
| Main limitation | Hash collisions may occur | Character patterns do not always correspond to meaningful chemical substructures |

For Morgan fingerprints, each column represents a hashed molecular-environment bit. For SMILES n-grams, each column represents a character sequence in the learned vocabulary.

The rows correspond to molecules, so both approaches can be evaluated using the same train, validation and test partitions. This enables a more controlled comparison of their predictive performance.


In [ ]:
print("Number of molecules")
print("Train:", len(y_train))
print("Validation:", len(y_valid))
print("Test:", len(y_test))

print("\nMorgan fingerprints")
print("Train:", X_train_morgan.shape)
print("Validation:", X_valid_morgan.shape)
print("Test:", X_test_morgan.shape)

print("\nSMILES n-grams")
print("Train:", X_train_text.shape)
print("Validation:", X_valid_text.shape)
print("Test:", X_test_text.shape)

### Distribution of molecular features

To better understand the two molecular representations, we examine the distribution of the number of active features per molecule.

For Morgan fingerprints, this corresponds to the number of active bits among the 2,048 available positions. For SMILES character n-grams, it corresponds to the number of distinct n-grams present in each molecule's representation.

These histograms help us understand how many features are typically used to represent a molecule and how much this number varies across the dataset. However, the two quantities are not directly comparable because they are based on different feature-generation methods.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse

# Count active features per molecule
morgan_active = np.asarray(X_train_morgan.sum(axis=1)).ravel()

text_active = np.asarray(
    X_train_text.getnnz(axis=1)
    if sparse.issparse(X_train_text)
    else np.count_nonzero(X_train_text, axis=1)
).ravel()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(morgan_active, bins=30)
axes[0].set_title("Morgan fingerprints")
axes[0].set_xlabel("Number of active bits")
axes[0].set_ylabel("Number of molecules")

axes[1].hist(text_active, bins=30)
axes[1].set_title("SMILES character n-grams")
axes[1].set_xlabel("Number of present n-grams")
axes[1].set_ylabel("Number of molecules")

plt.tight_layout()
plt.show()